# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [32]:
# load enviroment variables
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [33]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [34]:
import os, sys
from glob import glob


sys.path.append(os.getenv('SRC_DIR'))

# load variable price_data
price_data = os.getenv('PRICE_DATA')

stock_parquet_file = glob(os.path.join(price_data, "**/*.parquet"), recursive = True)

dd_px = dd.read_parquet(stock_parquet_file).set_index("ticker")

print(stock_parquet_file)


['../../05_src/data/prices/HYUP/HYUP_2009/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2009/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2014/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2014/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2013/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2013/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2012/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2012/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2015/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2015/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2010/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2010/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2017/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2017/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2019/part.0.parquet', '../../05_src/data/prices/HYUP/HYUP_2019/part.1.parquet', '../../05_src/data/prices/HYUP/HYUP_2018/part.0.parquet', '../../05_src

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [35]:
# add lags for variables close and adj_close.
dd_fit = dd_px.groupby("ticker", group_keys = False).apply(
    lambda x : x.assign(Close_lag_1 = x['Close'].shift(1))
)

dd_rets = dd_fit.groupby("ticker", group_keys = False).apply(
    lambda x : x.assign(Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

dd_feat = dd_rets.assign(
    Returns = lambda x : x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x : x['High'] - x['Low']
)

dd_feat.compute()

/var/folders/vf/yc4tnh9j5n7fyl3t8t8nff980000gp/T/ipykernel_4756/1853728416.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_fit = dd_px.groupby("ticker", group_keys = False).apply(
/var/folders/vf/yc4tnh9j5n7fyl3t8t8nff980000gp/T/ipykernel_4756/1853728416.py:6: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_rets = dd_fit.groupby("ticker", group_keys = False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
ticker,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2004-12-27,28.000000,29.230000,27.690001,28.670000,26.258089,651600.0,ZEUS.csv,2004,27.670000,25.342215,0.036140,1.539999
ZEUS,2004-12-28,28.900000,30.200001,28.700001,29.900000,27.384602,529400.0,ZEUS.csv,2004,28.670000,26.258089,0.042902,1.500000
ZEUS,2004-12-29,30.280001,30.299999,29.030001,29.070000,26.624434,355300.0,ZEUS.csv,2004,29.900000,27.384602,-0.027759,1.269999


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [36]:
# Convert to pandas dataframe
pd_feat = dd_feat.compute()

# moving average of returns using a window of 10days
pd_feat['Returns_MA_10'] = (
    pd_feat.groupby('ticker')['Returns']
    .transform(lambda x : x.rolling(window = 10, min_periods = 1).mean())
)

print(pd_feat)


             Date       Open       High        Low      Close  Adj Close  \
ticker                                                                     
A      1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
A      1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
A      1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
A      1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
A      1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   
...           ...        ...        ...        ...        ...        ...   
ZEUS   2004-12-27  28.000000  29.230000  27.690001  28.670000  26.258089   
ZEUS   2004-12-28  28.900000  30.200001  28.700001  29.900000  27.384602   
ZEUS   2004-12-29  30.280001  30.299999  29.030001  29.070000  26.624434   
ZEUS   2004-12-30  28.450001  28.450001  25.330000  26.170000  23.968391   
ZEUS   2004-12-31  25.940001  27.420000  25.940001  26.510000  24.279804   

           

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

> No it wasn't neccessary to convert to pandas

> if data is small, converting to Pandas is fine because its simpler code and fast.

> If data is large, its better to use Dask to avoid memory issues.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.